### Sonification individuelle des stations — STRAUSS

**Données : 24 juillet 2025**

#### Objectif et différence avec le notebook 07

Dans le **notebook 07**, il y a une carte animée combinant les précipitations de 16 pluviomètres et les concentrations d'oxygène dissous de 4 sondes dans la Senne.

Chaque station est représentée individuellement sur la carte, mais la bande sonore est obtenue en mélangeant deux fichiers audio déjà créés : un pour la pluie et un pour l'oxygène.

Cette première approche permet d'observer et d'écouter les deux phénomènes simultanément, **sans nécessairement pouvoir distinguer les stations à l'écoute**.

Dans le **notebook 08**, le travail vas plus loin: explorer une sonification où chaque station possède sa propre source sonore.

L'objectif est de tester si des caractéristiques sonores différentes (hauteur, timbre, position stéréo) permettent :

- de reconnaître les stations à l'écoute ;
- de percevoir les variations des mesures de chaque station ;
- de comparer les évolutions entre plusieurs stations.

Comme dans le notebook 07, les 24 heures du 24 juillet 2025 seront représentées en **60 secondes**.

Les correspondances entre données et sons seront considérées comme des choix expérimentaux, dont l'efficacité devra être évaluée.

#### Expérience 1 — Les quatre sondes d'oxygène

Premiere étape: les quatre sondes d'oxygène dissous (Q01, Q02, Q03 et Q04).

Chaque sonde sera associée à une voix sonore distincte, dont la hauteur évoluera selon les concentrations d'oxygène mesurées.
Test: différencier les sondes par leur position dans l'espace stéréo.

Une fois cette première expérience réalisée, même chose pour les 16 pluviomètres, en explorant une représentation des précipitations par des impulsions sonores. Le rôle de ce notebook est d'aider dans le choix d'une sonification pertinente pour les 16 événements de ton expérience finale.

### Station oxygène 

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Audio, display

from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss.score import Score
from strauss.generator import Synthesizer

# Charger les données déjà préparées
df = pd.read_csv("../data/24_juillet_2025_donnees_preparees.csv")

df["date"] = pd.to_datetime(df["Date/heure locale (UTC+2)"])

stations = ["Q01", "Q02", "Q03", "Q04"]

# Même durée pour toutes les stations
duree = 60
temps = np.linspace(0, 1, len(df))

print("Nombre de mesures :", len(df))
print("Stations :", stations)

Representation de chaque sonde par une source sonore independent.  Une référence commune de 10 mg/L est utilisé pour normaliser les concentrations (à voir si la garder)

Preparation

In [ ]:
# Normaliser les concentrations avec la même référence
reference_oxygene = 10.0

oxygene_normalise = {}

for station in stations:
    valeurs = pd.to_numeric(df[station], errors="coerce")

    oxygene_normalise[station] = np.clip(
        valeurs.to_numpy(dtype=float) / reference_oxygene,
        0, 1
    )

# Positions stéréo provisoires
positions = {
    "Q01": 0.25,
    "Q02": 0.42,
    "Q03": 0.58,
    "Q04": 0.75
}

print("Données préparées")

Sonification des 4 stations

In [ ]:
sons_oxygene = {}

for station in stations:

    # Une source sonore pour chaque station
    data_son = {
        "pitch": 1.0,
        "time_evo": temps,
        "pitch_shift": oxygene_normalise[station],
        "volume": 0.35,
        "azimuth": positions[station],
        "polar": 0.5
    }

    sources = Objects(data_son.keys())
    sources.fromdict(data_son)
    sources.apply_mapping_functions()

    generator = Synthesizer()
    generator.load_preset("pitch_mapper")

    score = Score([["A2"]], duree)

    soni = Sonification(score, sources, generator, "stereo")
    soni.render()

    sons_oxygene[station] = soni

    print("Sonification terminée :", station)

Ecouter la sonification

In [ ]:
# Écouter chaque station pour comparer les sons
for station in stations:
    print("Station :", station)
    sons_oxygene[station].notebook_display()

#### Écouter les quatre stations ensemble

Je mélange les quatre pistes pour vérifier si leurs variations restent perceptibles lorsqu'elles sont jouées simultanément.

Je conserve les positions stéréo et j'utilise une moyenne pour limiter le risque de saturation.

In [ ]:
# Récupérer les canaux audio de STRAUSS
def extraire_audio(sonification):
    streams = list(sonification.out_channels.values())

    audio = np.stack(
        [np.asarray(s.values, dtype=float) for s in streams],
        axis=0
    )

    frequence = int(streams[0].samprate)
    return audio, frequence


pistes = []

for station in stations:
    piste, frequence = extraire_audio(sons_oxygene[station])
    pistes.append(piste)

# Mélanger les quatre stations
mix_oxygene = np.mean(pistes, axis=0)

# Écouter sans modifier automatiquement le volume
display(Audio(mix_oxygene, rate=frequence, normalize=False))

In [ ]:
from scipy.io.wavfile import write

# Créer le dossier audio
dossier_audio = Path("../audio")
dossier_audio.mkdir(parents=True, exist_ok=True)

fichier_oxygene = dossier_audio / "08_oxygene_stations.wav"

# Préparer les deux canaux pour le WAV
audio_wav = np.nan_to_num(mix_oxygene.T)
audio_wav = np.clip(audio_wav, -1, 1)

write(fichier_oxygene, frequence, audio_wav.astype(np.float32))

print("Fichier enregistré :", fichier_oxygene)

### La pluie 

#### Expérience 2 — Les 16 pluviomètres

Je teste maintenant une sonification individuelle des 16 pluviomètres.

Contrairement à la sonification globale de la pluie, chaque pluviomètre possède sa propre source sonore.

Je cherche à représenter les précipitations par des impulsions : une mesure sans pluie reste silencieuse, tandis qu'une mesure positive déclenche un son.

L'intensité des précipitations contrôle la hauteur et le volume du son. Cette représentation reste exploratoire.

### Sonification des 16 pluviometres

In [ ]:
# Récupérer les 16 pluviomètres
colonnes_pluie = list(df.columns[1:17])

# Normaliser les précipitations
reference_pluie = 10.0

pluie_normalisee = {}

for capteur in colonnes_pluie:
    valeurs = pd.to_numeric(df[capteur], errors="coerce")

    pluie_normalisee[capteur] = np.clip(
        valeurs.to_numpy(dtype=float) / reference_pluie,
        0, 1
    )

# Répartir les stations dans l'espace stéréo
positions_pluie = np.linspace(0.1, 0.9, len(colonnes_pluie))

print("Nombre de pluviomètres :", len(colonnes_pluie))

In [ ]:
sons_pluie = {}

for i, capteur in enumerate(colonnes_pluie):

    intensite = pluie_normalisee[capteur]

    # Pas de son quand il ne pleut pas
    volume = np.where(intensite > 0, 0.25, 0.0)

    data_son = {
        "pitch": 1.0,
        "time_evo": temps,
        "pitch_shift": intensite,
        "volume": volume,
        "azimuth": float(positions_pluie[i]),
        "polar": 0.5
    }

    sources = Objects(data_son.keys())
    sources.fromdict(data_son)
    sources.apply_mapping_functions()

    generator = Synthesizer()
    generator.load_preset("pitch_mapper")

    score = Score([["A2"]], duree)

    soni = Sonification(score, sources, generator, "stereo")
    soni.render()

    sons_pluie[capteur] = soni

    print("Sonification terminée :", capteur)

### Créer les sons de pluie

Pour ce premier essai, j'utilise encore le synthétiseur STRAUSS `pitch_mapper`.

Le volume est nul lorsqu'il ne pleut pas. Lorsqu'une précipitation est mesurée, le son devient audible.

Il s'agit d'un premier test de sons intermittents, pas encore d'une véritable synthèse par impulsions courtes. Je pourrai ensuite comparer cette approche à des sons percussifs.

In [ ]:
sons_pluie = {}

for i, capteur in enumerate(colonnes_pluie):

    intensite = pluie_normalisee[capteur]

    # Pas de son quand il ne pleut pas
    volume = np.where(intensite > 0, 0.25, 0.0)

    data_son = {
        "pitch": 1.0,
        "time_evo": temps,
        "pitch_shift": intensite,
        "volume": volume,
        "azimuth": float(positions_pluie[i]),
        "polar": 0.5
    }

    sources = Objects(data_son.keys())
    sources.fromdict(data_son)
    sources.apply_mapping_functions()

    generator = Synthesizer()
    generator.load_preset("pitch_mapper")

    score = Score([["A2"]], duree)

    soni = Sonification(score, sources, generator, "stereo")
    soni.render()

    sons_pluie[capteur] = soni

    print("Sonification terminée :", capteur)

Ecouter la sonification

In [ ]:
pistes_pluie = []

for capteur in colonnes_pluie:
    piste, freq = extraire_audio(sons_pluie[capteur])

    if freq != frequence:
        raise ValueError("Fréquences audio différentes")

    pistes_pluie.append(piste)

# Mélanger les 16 sources
mix_pluie = np.mean(pistes_pluie, axis=0)

display(Audio(mix_pluie, rate=frequence, normalize=False))

In [ ]:
# Vérifier le volume du mélange
print("Amplitude maximale :", np.max(np.abs(mix_pluie)))

# Écouter avec une amplification automatique pour le test
display(Audio(mix_pluie, rate=frequence, normalize=True))